In [1]:
from bs4 import BeautifulSoup, Comment
import pandas as pd

import requests
import time
import csv
from io import StringIO

import sys
from pathlib import Path

pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, BASE_URL, HTTP_HEADER
import utils

### [Basketball-Reference Bot Policy](https://www.sports-reference.com/bot-traffic.html)

Basketball reference enforces a 1 day block if you are caught accessing the site more than 20 times a second.  All cells that run loops which collect the data should have `time.sleep(x)` in the loop where x $\geq3$.  In what follows we use `time.sleep(3.3)` for good measure.  Currently the cell should take about 

#### COMMENT OUT THIS CELL IF YOU HAVE ALREADY DOWNLOADED THE DATA AND NO CHANGES HAVE BEEN MADE TO `utils.scrape_per_game_tables()`

In [2]:
utils.scrape_per_game_tables(START_YEAR,END_YEAR)

Scraping data for the 1980-1981 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1981-1982 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1982-1983 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1983-1984 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1984-1985 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1985-1986 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1986-1987 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1987-1988 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1988-1989 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1989-1990 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1990-1991 season...
Adding team win/losses...
Ad

when we import the datafram from the csv it is important to add a comment keyword or it will mislabel the columns using the descriptive comment instead of the names.

Example for the 2005-2006 season

In [21]:
season = '2005-2006'
df = pd.read_csv(str(ROOT)+f'/data/{season}_season/per_game_stats_{season}.csv',encoding='utf-8',comment='#')

In [22]:
df.head(10)

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
0,2005,45.0,37.0,Kobe Bryant,27.0,LAL,SG,80.0,80.0,41.0,12.2,27.2,0.450,2.3,6.5,0.347,10.0,20.7,0.482,0.491,8.7,10.2,0.850,0.9,4.4,5.3,4.5,1.8,0.4,3.1,2.9,35.4,0.386
1,2005,38.0,44.0,Allen Iverson,30.0,PHI,PG,72.0,72.0,43.1,11.3,25.3,0.447,1.0,3.1,0.323,10.3,22.2,0.465,0.467,9.4,11.5,0.814,0.6,2.6,3.2,7.4,1.9,0.1,3.4,1.7,33.0,0.001
2,2005,50.0,32.0,LeBron James,21.0,CLE,SF,79.0,79.0,42.5,11.1,23.1,0.480,1.6,4.8,0.335,9.5,18.3,0.518,0.515,7.6,10.3,0.738,0.9,6.1,7.0,6.6,1.6,0.8,3.3,2.3,31.4,0.550
3,2005,42.0,40.0,Gilbert Arenas,24.0,WAS,PG,80.0,80.0,42.3,9.3,20.9,0.447,2.5,6.8,0.369,6.8,14.1,0.485,0.507,8.2,10.0,0.820,0.7,2.8,3.5,6.1,2.0,0.3,3.7,3.6,29.3,0.000
4,2005,52.0,30.0,Dwyane Wade,24.0,MIA,SG,75.0,75.0,38.6,9.3,18.8,0.495,0.2,1.0,0.171,9.1,17.8,0.513,0.499,8.4,10.7,0.783,1.4,4.3,5.7,6.7,1.9,0.8,3.6,2.9,27.2,0.070
5,2005,33.0,49.0,Paul Pierce,28.0,BOS,SF,79.0,79.0,39.0,8.7,18.5,0.471,1.4,4.0,0.354,7.3,14.5,0.503,0.509,7.9,10.3,0.772,1.0,5.7,6.7,4.7,1.4,0.4,3.5,2.8,26.8,0.000
6,2005,60.0,22.0,Dirk Nowitzki,27.0,DAL,PF,81.0,81.0,38.1,9.3,19.3,0.480,1.4,3.3,0.406,7.9,16.0,0.496,0.515,6.7,7.4,0.901,1.4,7.6,9.0,2.8,0.7,1.0,1.9,2.0,26.6,0.435
7,2005,44.0,38.0,Carmelo Anthony,21.0,DEN,SF,80.0,80.0,36.8,9.5,19.7,0.481,0.5,1.9,0.243,9.0,17.8,0.506,0.493,7.2,8.9,0.808,1.5,3.4,4.9,2.7,1.1,0.5,2.7,2.9,26.5,0.000
8,2005,40.0,42.0,Michael Redd,26.0,MIL,SG,80.0,80.0,39.1,8.5,19.0,0.450,2.0,5.2,0.395,6.5,13.8,0.471,0.504,6.3,7.1,0.877,1.0,3.3,4.3,2.9,1.2,0.1,2.1,2.0,25.4,0.000
9,2005,35.0,47.0,Ray Allen,30.0,SEA,SG,78.0,78.0,38.7,8.7,19.2,0.454,3.4,8.4,0.412,5.3,10.9,0.486,0.544,4.2,4.6,0.903,0.9,3.3,4.3,3.7,1.3,0.2,2.4,1.9,25.1,0.000


Note that players who have been traded during the season show up multiple times.  One for each team as well as an aggregate row where the team is listed as "#TM" and # is the number of teams they have been on. We have made the team wins/losses for these rows NaNs.  Here are the people who were on three teams.

In [26]:
df.loc[df['Team']=='3TM']

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
448,2005,NaN,NaN,Stephen Graham,23.0,3TM,SG,22.0,0.0,8.0,1.0,2.5,0.389,0.1,0.5,0.2,0.9,2.0,0.432,0.407,0.6,0.7,0.933,0.2,1.0,1.2,0.3,0.2,0.1,0.6,1.4,2.6,0.0
532,2005,NaN,NaN,Josh Davis,25.0,3TM,PF,6.0,0.0,2.8,0.3,1.2,0.286,0.0,0.3,0.0,0.3,0.8,0.400,0.286,0.3,0.5,0.667,0.5,0.2,0.7,0.2,0.3,0.0,0.2,0.5,1.0,0.0
540,2005,NaN,NaN,John Thomas,30.0,3TM,C,16.0,0.0,7.9,0.4,0.9,0.400,0.0,0.0,NaN,0.4,0.9,0.400,0.400,0.2,0.3,0.750,0.6,0.6,1.3,0.1,0.2,0.1,0.5,1.1,0.9,0.0


Now we import all of the data into a single pandas dataframe

In [27]:
df_tot = utils.player_stat_dataframe(START_YEAR,END_YEAR)

In [34]:
df_tot.sample(10)

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
6228,1995,36.0,46.0,Latrell Sprewell,25.0,GSW,SG,78.0,78.0,39.3,6.6,15.4,0.428,1.2,3.6,0.323,5.4,11.8,0.461,0.466,4.5,5.7,0.789,1.6,3.3,4.9,4.2,1.6,0.6,2.8,1.9,18.9,0.0
1522,1984,40.0,42.0,Cliff Robinson,24.0,WSB,PF,60.0,37.0,31.2,7.0,14.9,0.471,0.0,0.1,0.333,7.0,14.9,0.471,0.472,2.6,3.6,0.742,2.4,6.8,9.1,2.5,0.9,0.8,2.7,3.1,16.7,0.0
21915,2021,51.0,31.0,Semi Ojeleye,27.0,MIL,PF,20.0,0.0,15.4,0.9,3.5,0.257,0.6,2.1,0.268,0.4,1.5,0.241,0.336,0.5,0.7,0.769,1.3,1.7,2.9,0.3,0.3,0.3,0.3,1.3,2.9,0.0
19221,2017,58.0,24.0,Chris Boucher,25.0,GSW,PF,1.0,0.0,1.0,0.0,1.0,0.000,0.0,1.0,0.000,0.0,0.0,NaN,0.000,0.0,0.0,NaN,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22580,2022,37.0,45.0,Johnny Juzang,21.0,UTA,SG,18.0,0.0,12.9,1.9,5.8,0.337,0.8,3.5,0.238,1.1,2.3,0.488,0.409,0.1,0.2,0.500,0.4,1.7,2.2,0.4,0.2,0.2,0.2,0.7,4.8,0.0
16257,2013,62.0,20.0,Manu GinÃ³bili,36.0,SAS,SG,68.0,3.0,22.8,4.3,9.2,0.469,1.3,3.8,0.349,3.0,5.4,0.553,0.541,2.4,2.8,0.851,0.4,2.5,3.0,4.3,1.0,0.3,2.0,1.9,12.3,0.0
16857,2014,38.0,44.0,Trey Burke,22.0,UTA,PG,76.0,43.0,30.1,4.9,13.2,0.368,1.6,5.1,0.318,3.2,8.1,0.400,0.430,1.4,1.9,0.752,0.4,2.3,2.7,4.3,0.9,0.2,1.6,1.6,12.8,0.0
10693,2003,54.0,28.0,Elden Campbell,35.0,DET,C,65.0,27.0,13.7,2.1,4.8,0.439,0.0,0.0,NaN,2.1,4.8,0.439,0.439,1.3,1.9,0.685,0.8,2.4,3.2,0.7,0.3,0.8,1.0,2.1,5.6,0.0
23336,2023,46.0,36.0,Gui Santos,21.0,GSW,PF,23.0,0.0,8.3,1.2,2.4,0.509,0.4,1.2,0.370,0.8,1.2,0.643,0.600,0.7,0.7,0.941,0.7,1.4,2.1,0.6,0.2,0.1,0.3,1.0,3.6,0.0
7519,1997,41.0,41.0,Matt Maloney,26.0,HOU,PG,78.0,78.0,28.4,3.1,7.5,0.408,1.6,4.4,0.364,1.4,3.1,0.471,0.515,0.8,1.0,0.833,0.2,1.6,1.8,2.8,0.8,0.1,1.4,1.3,8.6,0.0
